# TN0.1 — pipeline của mình có giống pipeline gốc không

TN0 đã dựng lại được kết quả MobiVital bằng **code của họ**. Nhưng code đó chỉ chạy
được LSTM: dòng `mobivital_gen.py:152` ghi cứng `LSTMMultiStep(...)`, không nạp được TCN.

Nên phải viết bộ chọn kênh riêng — `src/scoring.py` — nhận model bất kỳ.

Notebook này kiểm tra bộ chọn kênh đó có **giống hệt** của MobiVital không.

## Cách kiểm

```
checkpoint LSTM cua MobiVital (lstm_pred_tripod_0.9.pth)
        |
        +---> mobivital_gen.py  cua ho     ->  results/TN0b.txt   (da co tu tn0.ipynb)
        |
        +---> src/scoring.py    cua minh   ->  TN0_1.txt          (chay o notebook nay)
                        |
              phai TRUNG 537/537 dong
```

Cùng checkpoint, cùng dữ liệu, cùng thuật toán, không có gì ngẫu nhiên → phải ra y hệt.
Lệch một dòng là bộ chọn kênh viết sai.

## Vì sao chỉ cần một phép kiểm này

Trùng 537/537 thì chứng minh luôn ba thứ cùng lúc:

| | vì sao suy ra được |
|---|---|
| dữ liệu `by_user/*.npz` đúng | nếu khác CSV thì kênh chọn ra đã lệch |
| bộ chọn kênh đúng | 537/537 |
| hàm chấm điểm đúng | cùng con số |

## Chạy ở đâu

**CPU**, cùng loại máy với lúc sinh `results/TN0b.txt`.

Bước chọn kênh là `argmax`. Đã đo ở TN0: cùng checkpoint, bản chạy GPU và bản chạy CPU
chọn khác kênh ở **251/537** buổi ghi. Muốn so `537/537` thì phải cùng thiết bị.

Từ TN1 trở đi chạy GPU thoải mái — lúc đó so cấu hình với nhau, không so với số cũ.

Mất khoảng **40 phút**.

## 1. Nạp code và dữ liệu

In [ ]:
import os
import sys
import csv
import time

import numpy as np
import torch

# Tìm thư mục gốc dự án bằng cách đi ngược lên tới khi thấy src/.
# Chạy được dù đang đứng ở notebooks/ hay ở gốc repo (Colab dùng %cd gốc repo).
GOC = os.getcwd()
while not os.path.exists(GOC + "/src/mobivital_reference.py"):
    cha = os.path.dirname(GOC)
    if cha == GOC:
        raise RuntimeError("không tìm thấy thư mục gốc dự án")
    GOC = cha

sys.path.insert(0, GOC)
print("thư mục gốc:", GOC)

from src import mobivital_reference as mv
from src import scoring

# Ép CPU. Xem phần "Chạy ở đâu" ở trên.
torch.set_num_threads(os.cpu_count())
print("thiết bị :", "cpu")
print("số luồng :", torch.get_num_threads())
mv.info()

## 2. Nạp checkpoint LSTM của MobiVital

Đúng file họ phát hành trong repo, không sửa gì.

In [ ]:
duong_dan = mv.MOBIVITAL_DIR + "/checkpoints/lstm_pred_tripod_0.9.pth"

model = mv.new_lstm()
model.load_state_dict(torch.load(duong_dan, map_location="cpu"))
model.eval()

so_tham_so = sum(p.numel() for p in model.parameters())
print(duong_dan)
print("số tham số:", format(so_tham_so, ","))
print("thử forward:", tuple(model(torch.randn(4, 200)).shape), " <- phải là (4, 25)")

## 3. Chạy bộ chọn kênh của mình trên 537 buổi ghi GHIJ

`scoring.write_txt` làm đúng việc `mobivital_gen.py` làm, cho từng buổi ghi:

1. dựng **240 ứng viên** — 120 kênh phức × 2 phép (`abs`, `phase`)
2. lọc bằng `invert_detector`, còn khoảng 137
3. cắt **52 cửa sổ** mỗi ứng viên — 200 mẫu vào → 25 mẫu đáp án, trượt 25
4. LSTM dự báo, tính Pearson giữa dự báo và 25 mẫu thật **của chính ứng viên đó**
5. `argmax` tổng Pearson → chọn kênh

Bước 4 và 5 **không nhìn nhịp thở thật**. Đó là điểm chính của bài báo.

Khác duy nhất: đọc `by_user/*.npz` thay vì `np.genfromtxt` từng file CSV.

In [ ]:
os.makedirs(GOC + "/runs/tn0_1", exist_ok=True)
duong_dan_ra = GOC + "/runs/tn0_1/TN0_1.txt"

luc_bat_dau = time.time()
diem_cua_minh = scoring.write_txt(["G", "H", "I", "J"], model, duong_dan_ra,
                                  by_user_dir=GOC + "/data/processed/by_user")
so_phut = (time.time() - luc_bat_dau) / 60

print("xong sau %.0f phút" % so_phut)
so_dong = len(open(duong_dan_ra).readlines())
print("số dòng ghi ra:", so_dong)
print("điểm trung bình 537 buổi ghi:", repr(diem_cua_minh))

assert so_dong == 537, "phải ra 537 dòng, ra %d" % so_dong

## 4. So từng dòng với kết quả code MobiVital sinh ra

`results/TN0b.txt` sinh từ `notebooks/tn0.ipynb` — chạy `mobivital_gen.py` bản gốc với
đúng checkpoint này. Xem output trong notebook đó.

Hai file có **thứ tự dòng khác nhau**: code MobiVital duyệt bằng `os.listdir`, mình đọc
`by_user` theo thứ tự sắp xếp. Nên phải khớp theo **tên file**, không theo số dòng.

In [ ]:
def doc_lua_chon(duong_dan):
    bang = {}
    for ten_file, bin_so, phep, co_lat in csv.reader(open(duong_dan)):
        bang[ten_file] = (bin_so, phep)
    return bang

cua_mobivital = doc_lua_chon(GOC + "/results/TN0b.txt")
cua_minh = doc_lua_chon(duong_dan_ra)

print("số dòng   MobiVital:", len(cua_mobivital), " | mình:", len(cua_minh))
print("cùng tập tên file  :", set(cua_mobivital) == set(cua_minh))
print()

trung = 0
khac = []
for ten_file in cua_mobivital:
    if cua_minh[ten_file] == cua_mobivital[ten_file]:
        trung = trung + 1
    else:
        khac.append((ten_file, cua_mobivital[ten_file], cua_minh[ten_file]))

print("TRÙNG %d / %d" % (trung, len(cua_mobivital)))
print("khác  %d" % len(khac))
for dong in khac[:10]:
    print("   ", dong)

# Dừng notebook ngay nếu lệch, đừng để chạy tiếp rồi in ra kết luận sai.
assert set(cua_mobivital) == set(cua_minh), "hai file không cùng tập tên file"
assert trung == 537, "chỉ trùng %d/537" % trung
assert len(khac) == 0

### Xem thử vài dòng cạnh nhau

In [ ]:
print("%-32s %-12s %-12s" % ("buổi ghi", "MobiVital", "mình"))
for ten_file in list(cua_mobivital)[:8]:
    a = cua_mobivital[ten_file]
    b = cua_minh[ten_file]
    print("%-32s %-12s %-12s %s" % (ten_file[:32], "%s,%s" % a, "%s,%s" % b,
                                    "OK" if a == b else "LỆCH"))

## 5. So điểm

`evaluate.py` của MobiVital chấm `TN0b.txt` được **0.8221751511496864** — xem output
trong `notebooks/tn0.ipynb`.

In [ ]:
DIEM_MOBIVITAL = 0.8221751511496864

print("điểm MobiVital :", repr(DIEM_MOBIVITAL))
print("điểm của mình  :", repr(diem_cua_minh))
chenh = abs(diem_cua_minh - DIEM_MOBIVITAL)
print("chênh lệch     : %.2e" % chenh)
print("1 đơn vị làm tròn nhỏ nhất của float64 ở giá trị này: %.2e"
      % np.spacing(DIEM_MOBIVITAL))

# Cho phép lệch tối đa 10 đơn vị làm tròn cuối cùng của float64.
assert chenh < 10 * np.spacing(DIEM_MOBIVITAL), "lệch quá mức sai số làm tròn"

Chênh lệch nằm ở **chữ số thứ 16**, cỡ vài đơn vị làm tròn cuối cùng của `float64`.

Nguyên nhân: `evaluate.py` cộng dồn trong vòng lặp (`total_score += score`) rồi mới chia,
còn `scoring.py` dùng `np.mean`. **Khác thứ tự cộng, không khác thuật toán.**

## 6. Kiểm tra thêm — sắp xếp rồi so nguyên file

In [ ]:
a = sorted(open(duong_dan_ra).read().split())
b = sorted(open(GOC + "/results/TN0b.txt").read().split())

print("số dòng bằng nhau  :", len(a) == len(b))
print("nội dung giống hệt :", a == b)

assert a == b, "hai file khác nhau sau khi sắp xếp"
print()
print("TẤT CẢ ASSERT ĐỀU QUA — pipeline của mình giống pipeline gốc")

## Kết luận

Trùng 537/537 và điểm khớp tới chữ số thứ 15 ⇒ `src/scoring.py` cho ra **đúng cùng kết quả**
với code MobiVital khi dùng cùng checkpoint.

Từ đây các thí nghiệm sau chỉ cần thay model, mọi khâu còn lại giữ nguyên:

```python
model = models.tao_model("ds_tcn", revin=True)
scoring.score_users(["A", "B"], model)
```

Câu trả lời cho hội đồng:

> Bộ chọn kênh của chúng em cho kết quả trùng khớp hoàn toàn 537/537 với mã nguồn gốc
> khi dùng cùng checkpoint. Sau đó chỉ thay bộ dự báo LSTM bằng TCN, còn dữ liệu, cách
> chọn waveform và cách chấm điểm giữ cố định.